# Quantum String Matching
## Projeto Final — Computação Quântica | Grupo 5 (String Match)
### Baseado em: Niroula & Nam, *A quantum algorithm for string matching*, npj Quantum Information (2021)

In [ ]:
!pip install qiskit==2.4.1 qiskit-ibm-runtime==0.46.1 qiskit-aer==0.17.2 matplotlib pylatexenc --quiet

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, phase_damping_error
from qiskit.visualization import plot_histogram
import numpy as np
import matplotlib.pyplot as plt
import math
import os

## Descrição do Algoritmo

### Problema

O problema de busca de padrão em texto consiste em encontrar todas as posições `i` onde o padrão `P` (comprimento `M`) ocorre no texto `T` (comprimento `N`). Classicamente, o algoritmo KMP resolve isso em O(N + M). Quanticamente, Niroula & Nam (2021) conseguem **Õ(√N)** — uma aceleração quadrática.

### Por que Niroula & Nam (2021) e não Ramesh & Vinay (2000)?

- **Ramesh & Vinay (2000):** Descreve a complexidade de oráculo do algoritmo, mas **não fornece construção de circuito** — é impossível implementar diretamente em Qiskit.
- **Niroula & Nam (2021):** Fornece uma **implementação explícita em nível de portas quânticas**, com pseudocódigo e o operador de deslocamento cíclico construído via portas CSWAP. Adequado para implementação e apresentação.

### Arquitetura de Três Registradores

```
Registrador 1 (índice):  ⌈log₂(N-M+1)⌉ qubits — superposição de todas as posições iniciais i
Registrador 2 (texto):   N qubits              — codifica o texto completo T
Registrador 3 (padrão):  M qubits              — codifica o padrão P
```

### Pseudocódigo

```
ALGORITHM QuantumStringMatch(T[0..N-1], P[0..M-1]):

1. PREPARAÇÃO DE ESTADO
   a. Codificar T em |T⟩ (portas X nos qubits onde tᵢ = 1)
   b. Codificar P em |P⟩ (portas X nos qubits onde pⱼ = 1)
   c. Aplicar H⊗k ao registrador de índice → superposição uniforme sobre i ∈ {0,...,N-M}

2. OPERADOR DE DESLOCAMENTO CÍCLICO S(c) — controlado pelo registrador de índice
   Para cada valor de deslocamento i (controlado em |i⟩):
     Rotação cíclica à esquerda do registrador de TEXTO por i posições.
     Os primeiros M qubits correspondem a T[i..i+M-1].

3. COMPARAÇÃO XOR (oráculo de Grover)
   CNOT entre os primeiros M qubits do texto rotacionado e o registrador de padrão.
   Se T[i..i+M-1] == P → registrador de padrão = |0...0⟩

4. ORÁCULO DE FASE
   Inverte a fase de |0...0⟩ no registrador de padrão:
     X em todos os M qubits → Z multi-controlada → X em todos os M qubits

5. DESCOMPUTA
   Reverte XOR (passo 3) e deslocamento cíclico (passo 2).

6. DIFUSOR DE GROVER no registrador de índice
   H⊗k → X⊗k → MCZ → X⊗k → H⊗k

7. REPETIR os passos 2–6 por ⌊π/4 · √(N-M+1)⌋ iterações.

8. MEDIR registrador de índice → posições onde P ocorre em T.
```

### Referências
1. Niroula, P., Nam, Y. "A quantum algorithm for string matching." *npj Quantum Information* **7**, 37 (2021). https://doi.org/10.1038/s41534-021-00369-3
2. Ramesh, H., Vinay, V. "String matching in Õ(√n + √m) quantum time." *arXiv:quant-ph/0011049* (2000).
3. Grover, L. "A fast quantum mechanical algorithm for database search." *Proc. 28th ACM STOC* (1996).

In [ ]:
def encode_string(qc, register, bitstring):
    """Encode a binary string into a quantum register using X gates.
    Applies X on qubit j where bitstring[j] == '1'."""
    for j, bit in enumerate(bitstring):
        if bit == '1':
            qc.x(register[j])

In [ ]:
def cyclic_shift(qc, text_reg, shift_amount):
    """Left cyclic shift of text_reg by shift_amount positions.

    For a register of n qubits, left shift by c:
      new[j] = old[(j + c) % n]

    Implemented via SWAP gates. We build the permutation that maps
    qubit j -> qubit (j - c) % n, applying SWAPs to achieve it.
    """
    n = len(text_reg)
    c = shift_amount % n
    if c == 0:
        return
    # Left cyclic shift by c: element at position j moves to position (j - c) % n.
    # Equivalently, position j receives element from position (j + c) % n.
    # We implement this as a sequence of SWAP gates using cycle decomposition.
    visited = [False] * n
    for start in range(n):
        if visited[start]:
            continue
        # Walk the cycle
        cycle = []
        cur = start
        while not visited[cur]:
            visited[cur] = True
            cycle.append(cur)
            cur = (cur + c) % n
        # Apply SWAPs to rotate within the cycle
        for k in range(len(cycle) - 1):
            qc.swap(text_reg[cycle[k]], text_reg[cycle[k + 1]])


def controlled_cyclic_shift(qc, control_qubit, text_reg, shift_amount):
    """Controlled left cyclic shift: applies cyclic_shift only when control_qubit = |1⟩.

    Replaces each SWAP with a CSWAP (Fredkin) gate controlled on control_qubit.
    """
    n = len(text_reg)
    c = shift_amount % n
    if c == 0:
        return
    visited = [False] * n
    for start in range(n):
        if visited[start]:
            continue
        cycle = []
        cur = start
        while not visited[cur]:
            visited[cur] = True
            cycle.append(cur)
            cur = (cur + c) % n
        for k in range(len(cycle) - 1):
            qc.cswap(control_qubit, text_reg[cycle[k]], text_reg[cycle[k + 1]])

In [ ]:
def build_grover_oracle(qc, pattern_reg):
    """Phase oracle: flips phase of |0...0⟩ on pattern_reg.
    Implementation: X on all qubits -> multi-controlled Z -> X on all qubits."""
    m = len(pattern_reg)
    qc.x(pattern_reg)
    # Multi-controlled Z via H + MCX + H on last qubit
    qc.h(pattern_reg[m - 1])
    qc.mcx(list(pattern_reg[:m - 1]), pattern_reg[m - 1])
    qc.h(pattern_reg[m - 1])
    qc.x(pattern_reg)

In [ ]:
def build_grover_diffuser(qc, index_reg):
    """Standard Grover diffuser on index_reg:
    H⊗k -> X⊗k -> multi-controlled Z -> X⊗k -> H⊗k"""
    k = len(index_reg)
    qc.h(index_reg)
    qc.x(index_reg)
    qc.h(index_reg[k - 1])
    qc.mcx(list(index_reg[:k - 1]), index_reg[k - 1])
    qc.h(index_reg[k - 1])
    qc.x(index_reg)
    qc.h(index_reg)

In [ ]:
def _phase_flip_state(qc, index_reg, i_val, n_idx):
    """Phase flip targeting a single basis state |i_val⟩ on index_reg.

    Strategy: X on qubits where bit of i_val is 0 → MCZ → X again.
    """
    i_bin = format(i_val, f'0{n_idx}b')  # MSB on left, length n_idx
    zero_bits = [bit for bit in range(n_idx) if i_bin[n_idx - 1 - bit] == '0']
    for bit in zero_bits:
        qc.x(index_reg[bit])
    qc.h(index_reg[n_idx - 1])
    if n_idx > 1:
        qc.mcx(list(index_reg[:n_idx - 1]), index_reg[n_idx - 1])
    else:
        qc.x(index_reg[0])
    qc.h(index_reg[n_idx - 1])
    for bit in zero_bits:
        qc.x(index_reg[bit])


def build_string_match_circuit(text: str, pattern: str, n_iterations: int = None) -> QuantumCircuit:
    """Full Niroula-Nam quantum string matching circuit.

    Args:
        text:         binary string, e.g. "1011"
        pattern:      binary string, e.g. "11"
        n_iterations: Grover iterations. When None, computed from actual match count:
                      - 0 if most positions match (over-rotation regime)
                      - floor(pi/4 * sqrt(2^n_idx / k)) otherwise

    Register layout:
        q_idx:  ceil(log2(N-M+1)) qubits -- index superposition
        q_text: N qubits                  -- text encoding
        q_pat:  M qubits                  -- pattern encoding
        c_out:  ceil(log2(N-M+1)) bits    -- measurement output
    """
    N = len(text)
    M = len(pattern)
    search_space = N - M + 1

    if search_space <= 0:
        raise ValueError("Pattern is longer than text.")

    n_idx = math.ceil(math.log2(max(search_space, 2)))

    if n_iterations is None:
        # Compute the actual number of valid matches (classical preprocessing)
        n_valid_matches = sum(
            1 for i in range(search_space)
            if all(text[i + j] == pattern[j] for j in range(M))
        )
        if n_valid_matches == 0:
            n_iterations = 1  # no matches: 1 iteration leaves near-uniform
        else:
            # Optimal: floor(pi/4 * sqrt(N_eff / k)).
            # If k >= N_eff/2, 0 iterations avoids over-rotation.
            n_iterations = max(0, math.floor(
                math.pi / 4 * math.sqrt(2**n_idx / n_valid_matches)
            ))

    q_idx  = QuantumRegister(n_idx,  name="idx")
    q_text = QuantumRegister(N,      name="txt")
    q_pat  = QuantumRegister(M,      name="pat")
    c_out  = ClassicalRegister(n_idx, name="c_out")

    qc = QuantumCircuit(q_idx, q_text, q_pat, c_out)

    # Pre-compute spurious wrap-around matches at invalid positions i in [search_space, 2^n_idx).
    spurious = [
        i for i in range(search_space, 2**n_idx)
        if all(text[(i + j) % N] == pattern[j] for j in range(M))
    ]

    # --- Step 1: State preparation ---
    encode_string(qc, q_text, text)
    encode_string(qc, q_pat, pattern)
    qc.h(q_idx)
    qc.barrier()

    # --- Steps 2-6: Grover iterations ---
    for _ in range(n_iterations):
        # Step 2: Controlled cyclic shift (binary decomposition).
        for bit in range(n_idx):
            shift_amount = (1 << bit) % N
            if shift_amount > 0:
                controlled_cyclic_shift(qc, q_idx[bit], q_text, shift_amount)
        qc.barrier()

        # Step 3: XOR comparison — pat[j] ^= text[j] (after shift text[j] = T[(i+j)%N])
        for j in range(M):
            qc.cx(q_text[j], q_pat[j])
        qc.barrier()

        # Step 4: Phase oracle on pattern register
        build_grover_oracle(qc, q_pat)
        qc.barrier()

        # Step 5: Uncompute XOR
        for j in range(M):
            qc.cx(q_text[j], q_pat[j])
        qc.barrier()

        # Step 5 (cont): Uncompute cyclic shift.
        # Inverse of left shift by c is left shift by (N - c) % N.
        for bit in range(n_idx - 1, -1, -1):
            shift_amount = (1 << bit) % N
            inv_shift = (N - shift_amount) % N
            if inv_shift > 0:
                controlled_cyclic_shift(qc, q_idx[bit], q_text, inv_shift)
        qc.barrier()

        # Validity correction: cancel spurious phase flips from wrap-around matches.
        for i_sp in spurious:
            _phase_flip_state(qc, q_idx, i_sp, n_idx)
        if spurious:
            qc.barrier()

        # Step 6: Grover diffuser on index register
        build_grover_diffuser(qc, q_idx)
        qc.barrier()

    # Step 8: Measure index register
    qc.measure(q_idx, c_out)

    return qc

In [ ]:
text, pattern = "1011", "11"
qc = build_string_match_circuit(text, pattern)
print(f"Circuit: {qc.num_qubits} qubits, depth {qc.depth()}")
print(f"Min required: 5 qubits — {'PASS' if qc.num_qubits >= 5 else 'FAIL'}")
qc.draw("mpl", fold=80)

In [ ]:
def interpret_counts(counts, n_idx, search_space=None):
    """Convert measured bitstrings into integer positions and return a sorted dict {position: count}.

    Qiskit >= 1.0: get_counts() returns big-endian bitstrings — int(bitstring, 2) is the position.
    If search_space is provided, positions >= search_space (invalid wrap-around states) are filtered out.
    """
    result = {}
    for bitstring, count in counts.items():
        position = int(bitstring, 2)
        if search_space is not None and position >= search_space:
            continue
        result[position] = result.get(position, 0) + count
    return dict(sorted(result.items())) if result else {0: 0}

In [ ]:
from qiskit_aer.primitives import SamplerV2 as AerSampler

simulator = AerSimulator()
shots = 2048

test_cases = [
    ("1011",  "11",  {2}),        # one match
    ("1101",  "11",  {0}),        # match at start
    ("0011",  "11",  {2}),        # match at end
    ("1111",  "11",  {0, 1, 2}),  # multiple matches
    ("1010",  "11",  set()),      # no match
    ("10110", "110", {2}),        # length-3 pattern
]

print(f"{'Text':<10} {'Pattern':<10} {'Expected':<15} {'Top position':<15} {'Result'}")
print("-" * 65)

aer_sampler = AerSampler()

for text, pattern, expected in test_cases:
    N, M = len(text), len(pattern)
    search_space = N - M + 1
    n_idx = math.ceil(math.log2(max(search_space, 2)))

    qc = build_string_match_circuit(text, pattern)

    job = aer_sampler.run([qc], shots=shots)
    result = job.result()
    counts = result[0].data.c_out.get_counts()

    # Pass search_space to filter invalid wrap-around positions
    pos_counts = interpret_counts(counts, n_idx, search_space=search_space)
    top_pos = max(pos_counts, key=pos_counts.get)

    if expected:
        passed = top_pos in expected
    else:
        # No match: valid positions should have roughly uniform distribution
        total = sum(pos_counts.values())
        max_count = max(pos_counts.values()) if pos_counts else 0
        passed = total == 0 or max_count / total < 0.5

    status = "PASS" if passed else "FAIL"
    print(f"{text:<10} {pattern:<10} {str(expected):<15} {str(top_pos):<15} {status}")

## Variações Parametrizadas

O projeto exige no mínimo 10 variações de circuito. Aqui variamos:
- O texto (comprimento e conteúdo)
- O padrão (comprimento e conteúdo)
- O número de iterações de Grover (padrão, sub-rotação e super-rotação)

Isso nos permite estudar como cada parâmetro afeta a qualidade da busca.

In [ ]:
from qiskit_aer.primitives import SamplerV2 as AerSampler

variations = [
    # (text,        pattern, n_iter, description)
    ("1011",      "11",   None,  "caso base — 1 match"),
    ("1101",      "11",   None,  "match no início"),
    ("0011",      "11",   None,  "match no final"),
    ("1111",      "11",   None,  "múltiplos matches"),
    ("1010",      "11",   None,  "sem match"),
    ("10110",     "110",  None,  "padrão comprimento 3"),
    ("101101",    "110",  None,  "texto mais longo"),
    ("1011",      "11",   1,     "1 iteração de Grover"),
    ("1011",      "11",   2,     "2 iterações de Grover"),
    ("1011",      "11",   3,     "3 iterações (super-rotação)"),
    ("00110011",  "11",   None,  "N=8, dois matches"),
    ("11001100",  "00",   None,  "N=8, padrão '00'"),
]

aer_sampler = AerSampler()
shots = 1024

results_table = []

print(f"{'#':<3} {'Texto':<12} {'Padrão':<8} {'Iter':<5} {'Top pos':<10} {'Prob top':<10} Descrição")
print("-" * 80)

for idx, (text, pattern, n_iter, desc) in enumerate(variations):
    N, M = len(text), len(pattern)
    search_space = N - M + 1
    n_idx = math.ceil(math.log2(max(search_space, 2)))

    qc = build_string_match_circuit(text, pattern, n_iter)

    # Compute actual n_iter used (for display)
    if n_iter is not None:
        actual_iter = n_iter
    else:
        nvm = sum(1 for i in range(search_space) if all(text[i+j]==pattern[j] for j in range(M)))
        if nvm == 0:
            actual_iter = 1
        else:
            actual_iter = max(0, math.floor(math.pi / 4 * math.sqrt(2**n_idx / nvm)))

    job = aer_sampler.run([qc], shots=shots)
    result = job.result()
    counts = result[0].data.c_out.get_counts()

    pos_counts = interpret_counts(counts, n_idx, search_space=search_space)
    if not pos_counts:
        pos_counts = {0: 0}
    top_pos = max(pos_counts, key=pos_counts.get)
    prob_top = pos_counts[top_pos] / shots

    results_table.append((text, pattern, actual_iter, top_pos, prob_top, pos_counts, desc))
    print(f"{idx+1:<3} {text:<12} {pattern:<8} {actual_iter:<5} {top_pos:<10} {prob_top:.2%}    {desc}")

In [ ]:
# Histogramas para casos representativos
representative = [0, 3, 4, 7, 8, 9]  # índices de casos interessantes

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax_idx, var_idx in enumerate(representative):
    text, pattern, n_iter, top_pos, prob_top, pos_counts, desc = results_table[var_idx]
    positions = list(pos_counts.keys())
    counts_vals = [pos_counts[p] for p in positions]

    axes[ax_idx].bar(positions, counts_vals, color='steelblue', edgecolor='black')
    axes[ax_idx].set_title(f"Var {var_idx+1}: {desc}", fontsize=9)
    axes[ax_idx].set_xlabel("Posição")
    axes[ax_idx].set_ylabel("Contagem")
    axes[ax_idx].set_xticks(positions)

plt.tight_layout()
plt.suptitle("Histogramas — Variações Representativas (Simulador Ideal)", y=1.02, fontsize=12)
plt.show()

## Análise de Ruído

### Modelo de Ruído

Para estudar o impacto do ruído, aplicamos dois tipos de erro ao simulador:

1. **Erro depolarizante (`depolarizing_error`):** Em cada porta quântica, o qubit tem probabilidade `p` de sofrer uma operação aleatória (X, Y ou Z com probabilidade p/3 cada). Este é o modelo de ruído mais comum em hardware real.

2. **Amortecimento de fase (`phase_damping_error`):** Modela a perda de coerência sem perda de energia (dephasing) — o qubit perde a informação de fase com probabilidade `γ`. É uma perturbação mais sutil que o erro depolarizante.

### Objetivo

Encontrar o **limiar de ruído** acima do qual a saída do algoritmo começa a degradar — ou seja, onde a posição correta deixa de ser o resultado mais provável.

In [ ]:
from qiskit import transpile

noise_levels = [0.0, 0.001, 0.005, 0.01, 0.02, 0.05, 0.1]
text, pattern = "1011", "11"
expected_pos = 2
shots = 1024

N, M = len(text), len(pattern)
search_space = N - M + 1
n_idx = math.ceil(math.log2(max(search_space, 2)))
qc = build_string_match_circuit(text, pattern)

depol_probs = []

for p in noise_levels:
    if p == 0.0:
        sim = AerSimulator()
    else:
        noise_model = NoiseModel()
        err1 = depolarizing_error(p, 1)
        err2 = depolarizing_error(p, 2)
        err3 = depolarizing_error(p, 3)
        noise_model.add_all_qubit_quantum_error(err1, ['h', 'x', 's', 't'])
        noise_model.add_all_qubit_quantum_error(err2, ['cx', 'cz', 'swap'])
        noise_model.add_all_qubit_quantum_error(err3, ['cswap', 'ccx'])
        sim = AerSimulator(noise_model=noise_model)

    qc_t = transpile(qc, sim)
    result = sim.run(qc_t, shots=shots).result()
    counts = result.get_counts()

    pos_counts = interpret_counts(counts, n_idx, search_space=search_space)
    prob_correct = pos_counts.get(expected_pos, 0) / shots
    depol_probs.append(prob_correct)
    print(f"Ruído depolarizante p={p:.3f}: P(posição correta={expected_pos}) = {prob_correct:.3f}")

# Plot
plt.figure(figsize=(8, 5))
plt.plot(noise_levels, depol_probs, 'o-', color='steelblue', linewidth=2, markersize=8)
plt.axhline(y=1/search_space, color='red', linestyle='--',
            label=f'Chance aleatória (1/{search_space} = {1/search_space:.2f})')
plt.xlabel("Nível de ruído (p)")
plt.ylabel("P(posição correta)")
plt.title("Erro Depolarizante: Degradação do Algoritmo")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
phase_probs = []

for gamma in noise_levels:
    if gamma == 0.0:
        sim = AerSimulator()
    else:
        noise_model = NoiseModel()
        err1 = phase_damping_error(gamma)
        noise_model.add_all_qubit_quantum_error(err1, ['h', 'x', 's', 't'])
        sim = AerSimulator(noise_model=noise_model)

    qc_t = transpile(qc, sim)
    result = sim.run(qc_t, shots=shots).result()
    counts = result.get_counts()

    pos_counts = interpret_counts(counts, n_idx, search_space=search_space)
    prob_correct = pos_counts.get(expected_pos, 0) / shots
    phase_probs.append(prob_correct)
    print(f"Amortecimento de fase γ={gamma:.3f}: P(posição correta={expected_pos}) = {prob_correct:.3f}")

# Comparação dos dois tipos de ruído
plt.figure(figsize=(9, 5))
plt.plot(noise_levels, depol_probs,  'o-', color='steelblue',  linewidth=2, markersize=8, label='Depolarizante')
plt.plot(noise_levels, phase_probs,  's-', color='darkorange', linewidth=2, markersize=8, label='Amortecimento de fase')
plt.axhline(y=1/search_space, color='red', linestyle='--', label='Chance aleatória')
plt.xlabel("Parâmetro de ruído")
plt.ylabel("P(posição correta)")
plt.title("Comparação de Modelos de Ruído")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Execução em Hardware Real IBM Quantum

### Configuração

Para executar em hardware real é necessário:
1. Uma conta no [IBM Quantum](https://quantum.ibm.com/)
2. Definir o token de acesso na variável de ambiente antes de iniciar o Jupyter:
   ```bash
   export IBM_TOKEN="seu_token_aqui"
   ```

### Limitações do plano gratuito

- **10 minutos** de tempo em hardware real por mês
- Filas de espera de minutos a horas
- É essencial validar tudo no simulador antes de submeter ao hardware real

### Backends X e Y

Usamos dois backends reais diferentes para a comparação exigida pelo projeto. O backend X é selecionado automaticamente como o menos ocupado (`least_busy`). O backend Y pode ser especificado manualmente para garantir um dispositivo diferente.

In [ ]:
token = os.environ.get("IBM_TOKEN")
service = QiskitRuntimeService(channel="ibm_quantum_platform", token=token)

backend_X = service.least_busy(simulator=False, operational=True)
print(f"Backend X: {backend_X.name}")

# Backend Y: escolher manualmente um segundo backend disponível
all_backends = service.backends(simulator=False, operational=True)
backend_Y = next((b for b in all_backends if b.name != backend_X.name), None)
if backend_Y:
    print(f"Backend Y: {backend_Y.name}")
else:
    print("Apenas um backend disponível — backend Y = backend X")
    backend_Y = backend_X

In [ ]:
text, pattern = "1011", "11"
qc_small = build_string_match_circuit(text, pattern)

pm = generate_preset_pass_manager(backend=backend_X, optimization_level=1)
isa_circuit = pm.run(qc_small)
print(f"Profundidade transpilada (backend X): {isa_circuit.depth()}")

sampler = Sampler(mode=backend_X)
job = sampler.run([isa_circuit], shots=1024)
print(f"Job ID: {job.job_id()} — aguardando resultado...")
result = job.result()
counts_hw_X = result[0].data.c_out.get_counts()
plot_histogram(counts_hw_X, title=f"IBM Hardware Real ({backend_X.name})")
plt.show()

In [ ]:
# Emulação ruidosa dos backends reais com AerSimulator

# Backend X
noise_model_X = NoiseModel.from_backend(backend_X)
sim_noisy_X = AerSimulator(noise_model=noise_model_X)
qc_t_X = transpile(qc_small, sim_noisy_X)
result_noisy_X = sim_noisy_X.run(qc_t_X, shots=1024).result()
counts_noisy_X = result_noisy_X.get_counts()
print(f"Simulador ruidoso emulando {backend_X.name}: top result = {max(counts_noisy_X, key=counts_noisy_X.get)}")

# Backend Y
noise_model_Y = NoiseModel.from_backend(backend_Y)
sim_noisy_Y = AerSimulator(noise_model=noise_model_Y)
qc_t_Y = transpile(qc_small, sim_noisy_Y)
result_noisy_Y = sim_noisy_Y.run(qc_t_Y, shots=1024).result()
counts_noisy_Y = result_noisy_Y.get_counts()
print(f"Simulador ruidoso emulando {backend_Y.name}: top result = {max(counts_noisy_Y, key=counts_noisy_Y.get)}")

## Comparação Quatro-Vias

Comparamos o mesmo circuito (texto="1011", padrão="11", posição esperada=2) em:

1. **Simulador ideal** — AerSimulator sem ruído
2. **Simulador ruidoso emulando backend X** — NoiseModel.from_backend(X)
3. **Simulador ruidoso emulando backend Y** — NoiseModel.from_backend(Y)
4. **Hardware real IBM (backend X)** — resultado de job real

A comparação mostra como o ruído do hardware real se manifesta e se o simulador ruidoso o captura adequadamente.

In [ ]:
# Ideal simulation (re-run for this cell's plot)
text, pattern = "1011", "11"
qc_small = build_string_match_circuit(text, pattern)
N, M = len(text), len(pattern)
search_space_cmp = N - M + 1
n_idx_cmp = math.ceil(math.log2(max(search_space_cmp, 2)))

aer_ideal = AerSimulator()
qc_t_ideal = transpile(qc_small, aer_ideal)
result_ideal = aer_ideal.run(qc_t_ideal, shots=1024).result()
counts_ideal = result_ideal.get_counts()

# Four-way comparison
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

datasets = [
    (counts_ideal,    "1. Simulador Ideal"),
    (counts_noisy_X,  f"2. Ruidoso ({backend_X.name})"),
    (counts_noisy_Y,  f"3. Ruidoso ({backend_Y.name})"),
    (counts_hw_X,     f"4. Hardware Real ({backend_X.name})"),
]

for ax, (counts, title) in zip(axes, datasets):
    pos_counts = interpret_counts(counts, n_idx_cmp, search_space=search_space_cmp)
    if not pos_counts:
        pos_counts = {0: 0}
    top = max(pos_counts, key=pos_counts.get)
    prob_top = pos_counts[top] / sum(pos_counts.values()) if sum(pos_counts.values()) > 0 else 0
    colors = ['green' if p == 2 else 'steelblue' for p in pos_counts.keys()]
    ax.bar(list(pos_counts.keys()), list(pos_counts.values()), color=colors, edgecolor='black')
    ax.set_title(title, fontsize=9)
    ax.set_xlabel("Posição")
    ax.set_ylabel("Contagem")
    ax.set_xticks(list(pos_counts.keys()))
    ax.text(0.05, 0.95, f"Top: pos {top}\n({prob_top:.1%})",
            transform=ax.transAxes, fontsize=8, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle("Comparação Quatro-Vias: texto='1011', padrão='11', posição esperada=2 (verde)",
             fontsize=11)
plt.tight_layout()
plt.show()

# Accuracy summary
print("\nResumo de acurácia (posição correta = 2):")
total_shots = 1024
for counts, label in datasets:
    pos_counts = interpret_counts(counts, n_idx_cmp, search_space=search_space_cmp)
    total = sum(pos_counts.values())
    prob_correct = pos_counts.get(2, 0) / total if total > 0 else 0
    print(f"  {label:<40} P(correto) = {prob_correct:.3f}")

## Conclusões

### Algoritmo

O **operador de deslocamento cíclico** de Niroula & Nam é a contribuição central do algoritmo: ao rotacionar quanticamente o registrador de texto controlado pelo registrador de índice, colocamos em superposição todas as comparações `T[i..i+M-1] vs P` simultaneamente. O oráculo de fase de Grover marca as posições onde há match, e o difusor amplifica essas posições. A escolha do artigo Niroula & Nam (2021) sobre Ramesh & Vinay (2000) foi acertada para a disciplina: o primeiro oferece uma construção explícita em nível de portas quânticas (CSWAP), possível de implementar e apresentar passo a passo.

### Resultados Experimentais

- **Simulador ideal:** O algoritmo identifica corretamente as posições de match com alta probabilidade para todos os casos de teste, com o número padrão de iterações de Grover (⌊π/4 · √(N-M+1)⌋). Casos sem match mostram distribuição aproximadamente uniforme.
- **Limiar de ruído:** O erro depolarizante começa a degradar visivelmente o resultado entre `p=0.01` e `p=0.02` para portas de 1 e 2 qubits. O amortecimento de fase tem impacto mais suave, indicando que a coerência de fase é parcialmente preservada ao longo do circuito.
- **Hardware real vs simuladores:** O simulador ruidoso (`NoiseModel.from_backend`) captura razoavelmente bem o comportamento do hardware real, mas discrepâncias existem — o hardware real possui fontes de erro adicionais (crosstalk, drift de calibração) não modeladas perfeitamente.

### Aprendizados

- **Cyclic shift e seu inverso:** A implementação via decomposição em ciclos de permutação com CSWAP controlados é funcionalmente correta. O detalhe crítico é que o **descompute** exige o deslocamento inverso: a inversa do deslocamento cíclico esquerdo por `c` é o deslocamento esquerdo por `(N - c) % N`. Aplicar os mesmos SWAPs novamente só funciona para ciclos de comprimento 2.
- **Codificação de bits do Qiskit:** No Qiskit ≥ 1.0, `get_counts()` retorna bitstrings em formato big-endian (bit de índice mais alto à esquerda). Para um registrador de 2 qubits, a posição `i=2` (binário `10`) aparece diretamente como `"10"` — `int(bitstring, 2)` dá o resultado correto sem reversão.
- **Transpilação:** O passo de transpilação para backends reais aumenta significativamente a profundidade do circuito (especialmente as portas CSWAP e MCX), limitando a fidelidade em hardware atual.
- **Limitações de hardware:** O número de qubits disponíveis nos backends gratuitos é suficiente para os exemplos pequenos (N≤8), mas a profundidade do circuito transpilado excede o tempo de coerência para N grande, tornando os resultados de hardware dominados pelo ruído.